# SHA Input Geometry Probe

This notebook treats SHA-256 as a fixed round topology with switchable constant fields.
It measures both:

- coarse invariants under K-variation
- per-input trajectory shape under fixed topology


In [1]:
# Optional installs
# This notebook is stdlib-only. Nothing required.


In [2]:

"""
sha_input_geometry_probe.py
===========================

Purpose
-------
Probe SHA-256 as a fixed round topology with switchable constant fields and
measure how INPUT geometry propagates through the machine.

This script does two different jobs:

1. Constant-field sweep:
   Tests whether coarse invariants remain stable when K is changed.
   Measures:
     - Sziklai identity violations
     - final digest avalanche
     - final digest mean Hamming weight
     - INTERNAL trace difference vs standard K

2. Input-geometry probe:
   For a chosen base message, flips each input bit and records:
     - round-by-round state Hamming delta
     - first active round
     - peak round
     - total disturbance integral
     - carry-shadow distance by round
   Then groups similar input bits into trajectory families.

This is not a solver. It is a shape tracer.

No third-party dependencies are required.
Optional plotting can be added later if desired.

Run
---
python sha_input_geometry_probe.py

The script prints two reports:
  A. constant variant comparison
  B. input geometry probe for a selected message

You can modify CONFIG at the top.
"""

from __future__ import annotations

import hashlib
import itertools
import json
import math
import random
import statistics
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

MASK32 = 0xFFFFFFFF

# ============================================================================
# CONFIG
# ============================================================================

CONFIG = {
    "seed": 1337,
    "num_constant_sweep_messages": 48,
    "num_avalanche_pairs_per_variant": 48,
    "base_message": b"Dean Kulik / SHA input geometry probe / April 2026",
    "input_probe_bit_limit": 128,   # first N input bits to probe
    "rounds": 64,
    "write_json": True,
    "output_json": "sha_input_geometry_probe_results.json",
}

# ============================================================================
# STANDARD SHA-256 CONSTANTS
# ============================================================================

H0_STD = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K_STD = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5, 0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3, 0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC, 0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7, 0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13, 0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3, 0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5, 0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208, 0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

# ============================================================================
# PRIMES / ROOT-DERIVED ALTERNATE K TABLES
# ============================================================================

def primes_first(n: int) -> List[int]:
    out = []
    x = 2
    while len(out) < n:
        is_prime = True
        r = int(math.isqrt(x))
        for p in out:
            if p > r:
                break
            if x % p == 0:
                is_prime = False
                break
        if is_prime:
            out.append(x)
        x += 1
    return out

PRIMES64 = primes_first(64)

def frac_word_from_root(p: int, power: float) -> int:
    x = p ** power
    frac = x - math.floor(x)
    return int(frac * (2 ** 32)) & MASK32

def build_root_k(power: float) -> List[int]:
    return [frac_word_from_root(p, power) for p in PRIMES64]

# ============================================================================
# SHA-256 CORE
# ============================================================================

def rotr(x: int, n: int) -> int:
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return (x & y) ^ (~x & z)

def maj(x: int, y: int, z: int) -> int:
    return (x & y) ^ (x & z) ^ (y & z)

def big_sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def big_sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def small_sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def small_sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def add32(*xs: int) -> Tuple[int, int]:
    total = sum(xs)
    return total & MASK32, total >> 32

def hw32(x: int) -> int:
    return (x & MASK32).bit_count()

def hw256(words: Sequence[int]) -> int:
    return sum(hw32(w) for w in words)

def sha256_pad_single_block(msg: bytes) -> bytes:
    if len(msg) > 55:
        raise ValueError("This probe script uses single-block messages only (<=55 bytes).")
    bit_len = len(msg) * 8
    padded = msg + b"\x80"
    while len(padded) % 64 != 56:
        padded += b"\x00"
    padded += bit_len.to_bytes(8, "big")
    assert len(padded) == 64
    return padded

def words_from_block(block: bytes) -> List[int]:
    return [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

def build_schedule_single_block(msg: bytes) -> List[int]:
    block = sha256_pad_single_block(msg)
    W = words_from_block(block)
    for t in range(16, 64):
        w, _ = add32(small_sigma1(W[t-2]), W[t-7], small_sigma0(W[t-15]), W[t-16])
        W.append(w)
    return W

@dataclass
class RoundTrace:
    round_index: int
    state_in: Tuple[int, ...]
    state_out: Tuple[int, ...]
    T1: int
    T2: int
    T1_carry: int
    T2_carry: int
    sziklai_residual: int

@dataclass
class RunResult:
    H: Tuple[int, ...]
    digest_hex: str
    W: List[int]
    traces: List[RoundTrace]

def run_sha256_single_block(
    msg: bytes,
    K: Sequence[int] = K_STD,
    H0: Sequence[int] = H0_STD,
) -> RunResult:
    W = build_schedule_single_block(msg)
    a, b, c, d, e, f, g, h = H0
    traces: List[RoundTrace] = []

    for r in range(64):
        state_in = (a, b, c, d, e, f, g, h)

        T1, c1 = add32(h, big_sigma1(e), ch(e, f, g), K[r], W[r])
        T2, c2 = add32(big_sigma0(a), maj(a, b, c))

        new_a, _ = add32(T1, T2)
        new_e, _ = add32(d, T1)

        sziklai_residual = ((new_a - new_e) - (T2 - d)) & MASK32

        h = g
        g = f
        f = e
        e = new_e
        d = c
        c = b
        b = a
        a = new_a

        state_out = (a, b, c, d, e, f, g, h)
        traces.append(
            RoundTrace(
                round_index=r,
                state_in=state_in,
                state_out=state_out,
                T1=T1,
                T2=T2,
                T1_carry=c1,
                T2_carry=c2,
                sziklai_residual=sziklai_residual,
            )
        )

    H = []
    for x0, x in zip(H0, (a, b, c, d, e, f, g, h)):
        y, _ = add32(x0, x)
        H.append(y)

    digest_hex = "".join(f"{x:08x}" for x in H)
    return RunResult(H=tuple(H), digest_hex=digest_hex, W=W, traces=traces)

# ============================================================================
# K VARIANTS
# ============================================================================

def build_k_variants() -> Dict[str, List[int]]:
    return {
        "standard_cbrt_primes": list(K_STD),
        "all_zero": [0] * 64,
        "reversed_standard": list(reversed(K_STD)),
        "sqrt_primes": build_root_k(1/2),
        "cbrt_primes": build_root_k(1/3),
        "fourth_root_primes": build_root_k(1/4),
        "fifth_root_primes": build_root_k(1/5),
    }

# ============================================================================
# METRICS
# ============================================================================

def hamming_hex_digest(d1: str, d2: str) -> int:
    return bin(int(d1, 16) ^ int(d2, 16)).count("1")

def flip_message_bit(msg: bytes, bit_index: int) -> bytes:
    nbits = len(msg) * 8
    if not (0 <= bit_index < nbits):
        raise ValueError(f"bit_index {bit_index} outside message bit range 0..{nbits-1}")
    ba = bytearray(msg)
    byte_i = bit_index // 8
    bit_in_byte = 7 - (bit_index % 8)
    ba[byte_i] ^= (1 << bit_in_byte)
    return bytes(ba)

def random_single_block_message(rng: random.Random, min_len: int = 1, max_len: int = 55) -> bytes:
    n = rng.randint(min_len, max_len)
    return bytes(rng.getrandbits(8) for _ in range(n))

def round_state_delta_hw(t0: RoundTrace, t1: RoundTrace) -> int:
    return sum(hw32(a ^ b) for a, b in zip(t0.state_out, t1.state_out))

def round_carry_shadow_distance(t0: RoundTrace, t1: RoundTrace) -> int:
    # low-dimensional but useful "shadow"
    return abs(t0.T1_carry - t1.T1_carry) + abs(t0.T2_carry - t1.T2_carry)

# ============================================================================
# CONSTANT SWEEP
# ============================================================================

def constant_sweep_report(rng: random.Random) -> Dict[str, dict]:
    variants = build_k_variants()
    messages = [random_single_block_message(rng) for _ in range(CONFIG["num_constant_sweep_messages"])]

    report: Dict[str, dict] = {}

    # Use standard K as reference for internal trace differences.
    ref_runs = {m: run_sha256_single_block(m, K_STD) for m in messages}

    for name, K in variants.items():
        sziklai_violations = 0
        final_hw = []
        trace_delta_vs_std = []
        carry_shadow_delta_vs_std = []

        # run messages
        for m in messages:
            rr = run_sha256_single_block(m, K)
            ref = ref_runs[m]

            final_hw.append(sum(hw32(w) for w in rr.H))
            sziklai_violations += sum(1 for tr in rr.traces if tr.sziklai_residual != 0)

            per_round_state = [round_state_delta_hw(tr, ref_tr) for tr, ref_tr in zip(rr.traces, ref.traces)]
            per_round_carry = [round_carry_shadow_distance(tr, ref_tr) for tr, ref_tr in zip(rr.traces, ref.traces)]

            trace_delta_vs_std.append(sum(per_round_state) / len(per_round_state))
            carry_shadow_delta_vs_std.append(sum(per_round_carry) / len(per_round_carry))

        # avalanche within THIS K
        avalanches = []
        for _ in range(CONFIG["num_avalanche_pairs_per_variant"]):
            m = random_single_block_message(rng)
            bit = rng.randrange(len(m) * 8)
            m2 = flip_message_bit(m, bit)
            d1 = run_sha256_single_block(m, K).digest_hex
            d2 = run_sha256_single_block(m2, K).digest_hex
            avalanches.append(hamming_hex_digest(d1, d2))

        report[name] = {
            "sziklai_violations": sziklai_violations,
            "mean_final_hw": statistics.mean(final_hw),
            "stdev_final_hw": statistics.pstdev(final_hw),
            "mean_avalanche_bits": statistics.mean(avalanches),
            "stdev_avalanche_bits": statistics.pstdev(avalanches),
            # These two are the important corrections to "it worked the same".
            "mean_internal_state_delta_vs_standard": statistics.mean(trace_delta_vs_std),
            "mean_internal_carry_shadow_delta_vs_standard": statistics.mean(carry_shadow_delta_vs_std),
        }

    return report

# ============================================================================
# INPUT GEOMETRY PROBE
# ============================================================================

def probe_input_geometry(base_message: bytes, K: Sequence[int]) -> Dict[str, object]:
    if len(base_message) > 55:
        raise ValueError("base_message must fit in one SHA-256 block for this probe.")
    base_run = run_sha256_single_block(base_message, K)

    nbits = len(base_message) * 8
    limit = min(CONFIG["input_probe_bit_limit"], nbits)

    bit_records = []

    for bit in range(limit):
        m2 = flip_message_bit(base_message, bit)
        rr = run_sha256_single_block(m2, K)

        state_deltas = [round_state_delta_hw(a, b) for a, b in zip(base_run.traces, rr.traces)]
        carry_deltas = [round_carry_shadow_distance(a, b) for a, b in zip(base_run.traces, rr.traces)]

        active_rounds = [i for i, v in enumerate(state_deltas) if v != 0]
        first_round = active_rounds[0] if active_rounds else None
        peak_round = max(range(64), key=lambda i: state_deltas[i])
        peak_value = state_deltas[peak_round]
        integral = sum(state_deltas)
        final_digest_delta = hamming_hex_digest(base_run.digest_hex, rr.digest_hex)

        bit_records.append({
            "bit": bit,
            "byte": bit // 8,
            "bit_in_byte_msb": bit % 8,
            "first_active_round": first_round,
            "peak_round": peak_round,
            "peak_value": peak_value,
            "integral": integral,
            "final_digest_delta": final_digest_delta,
            "state_deltas_by_round": state_deltas,
            "carry_shadow_by_round": carry_deltas,
        })

    # Simple family grouping using a compact signature.
    # This is intentionally crude but useful.
    families: Dict[Tuple[int, int, int], List[int]] = defaultdict(list)
    for rec in bit_records:
        key = (
            rec["first_active_round"] if rec["first_active_round"] is not None else -1,
            rec["peak_round"],
            int(round(rec["integral"] / 16.0)),
        )
        families[key].append(rec["bit"])

    top_families = sorted(
        [{"signature": list(sig), "bits": bits, "count": len(bits)} for sig, bits in families.items()],
        key=lambda x: (-x["count"], x["signature"])
    )

    summary = {
        "base_message_ascii": base_message.decode("utf-8", errors="replace"),
        "base_message_hex": base_message.hex(),
        "base_digest": base_run.digest_hex,
        "num_bits_probed": limit,
        "mean_final_digest_delta": statistics.mean(r["final_digest_delta"] for r in bit_records),
        "mean_integral": statistics.mean(r["integral"] for r in bit_records),
        "mean_first_active_round": statistics.mean(r["first_active_round"] for r in bit_records if r["first_active_round"] is not None),
        "trajectory_families_top10": top_families[:10],
        "bit_records": bit_records,
    }
    return summary

# ============================================================================
# PRINT HELPERS
# ============================================================================

def print_constant_sweep(report: Dict[str, dict]) -> None:
    print("\n" + "=" * 88)
    print("A. CONSTANT SWEEP")
    print("=" * 88)
    print(
        f"{'variant':28s} "
        f"{'sziklai':>9s} "
        f"{'meanHW':>10s} "
        f"{'avalanche':>10s} "
        f"{'stateΔvsSTD':>12s} "
        f"{'carryΔvsSTD':>12s}"
    )
    print("-" * 88)
    for name, d in report.items():
        print(
            f"{name:28s} "
            f"{d['sziklai_violations']:9d} "
            f"{d['mean_final_hw']:10.2f} "
            f"{d['mean_avalanche_bits']:10.2f} "
            f"{d['mean_internal_state_delta_vs_standard']:12.2f} "
            f"{d['mean_internal_carry_shadow_delta_vs_standard']:12.2f}"
        )
    print("\nInterpretation:")
    print("  - If meanHW and avalanche remain near standard across variants, coarse invariants are topology-led.")
    print("  - If stateΔvsSTD and carryΔvsSTD are non-zero, the internal path is NOT 'the same'.")
    print("  - So 'changing K did not matter' can be true only for bulk metrics, not for full trajectory geometry.")

def print_input_probe(summary: Dict[str, object]) -> None:
    print("\n" + "=" * 88)
    print("B. INPUT GEOMETRY PROBE")
    print("=" * 88)
    print(f"base message : {summary['base_message_ascii']}")
    print(f"base digest  : {summary['base_digest']}")
    print(f"bits probed  : {summary['num_bits_probed']}")
    print(f"mean Δdigest : {summary['mean_final_digest_delta']:.2f} bits")
    print(f"mean integral: {summary['mean_integral']:.2f}")
    print(f"mean first r : {summary['mean_first_active_round']:.2f}")
    print("\nTop trajectory families (signature = [first_active_round, peak_round, rounded_integral/16]):")
    for fam in summary["trajectory_families_top10"]:
        print(f"  signature={fam['signature']} count={fam['count']:3d} bits={fam['bits'][:12]}{'...' if len(fam['bits'])>12 else ''}")

    print("\nFirst 12 bit probes:")
    for rec in summary["bit_records"][:12]:
        print(
            f"  bit={rec['bit']:3d} "
            f"byte={rec['byte']:2d} "
            f"msb_bit={rec['bit_in_byte_msb']} "
            f"first={rec['first_active_round']:2d} "
            f"peak=({rec['peak_round']:2d},{rec['peak_value']:3d}) "
            f"integral={rec['integral']:4d} "
            f"Δdigest={rec['final_digest_delta']:3d}"
        )

# ============================================================================
# MAIN
# ============================================================================

def main() -> None:
    rng = random.Random(CONFIG["seed"])

    sweep = constant_sweep_report(rng)
    print_constant_sweep(sweep)

    probe = probe_input_geometry(CONFIG["base_message"], K_STD)
    print_input_probe(probe)

    if CONFIG["write_json"]:
        payload = {
            "config": CONFIG,
            "constant_sweep": sweep,
            "input_geometry_probe": probe,
        }
        out_path = Path(CONFIG["output_json"])
        out_path.write_text(json.dumps(payload, indent=2))
        print(f"\nWrote JSON results to: {out_path.resolve()}")

if __name__ == "__main__":
    main()



A. CONSTANT SWEEP
variant                        sziklai     meanHW  avalanche  stateΔvsSTD  carryΔvsSTD
----------------------------------------------------------------------------------------
standard_cbrt_primes                 0     127.25     128.46         0.00         0.00
all_zero                             0     128.17     129.35       125.04         1.21
reversed_standard                    0     127.62     128.67       125.19         1.15
sqrt_primes                          0     126.19     126.25       124.87         1.10
cbrt_primes                          0     127.25     127.00         0.00         0.00
fourth_root_primes                   0     128.00     126.77       124.98         1.16
fifth_root_primes                    0     127.46     128.50       125.44         1.14

Interpretation:
  - If meanHW and avalanche remain near standard across variants, coarse invariants are topology-led.
  - If stateΔvsSTD and carryΔvsSTD are non-zero, the internal path is NOT 'th

TypeError: Object of type bytes is not JSON serializable

In [ ]:
main()
